# Aula 2 — O pipeline dos anúncios: texto e imagem

Na aula 1 montamos a tubulação: um tópico do Pub/Sub, uma assinatura que grava no BigQuery e outra que
grava no Cloud Storage. Agora a gente coloca dado de verdade correndo por ela.

O fio condutor é **um anúncio de imóvel**. Ele entra no pipeline e se divide em dois caminhos:

- o **texto** (preço, área, endereço, descrição) vai pelo Pub/Sub e chega ao BigQuery, de onde saem as
  camadas Bronze / Silver / Gold que vocês já sabem construir;
- a **imagem** vai para o Cloud Storage, e de lá é o Gemini que tira dela a informação que o texto não
  traz.

No fim, os dois caminhos se reencontram no BigQuery.

> **Pré-requisito**: ter rodado o `gcp-pubsub-v2.ipynb` (aula 1) neste mesmo projeto. É de lá que vêm o
> tópico `aula-pdm-anuncios`, a tabela `aula_pdm.anuncios` e o bucket do Cloud Storage.

## 0. Configuração

In [ ]:
%pip install --quiet requests pydantic pillow google-cloud-pubsub google-cloud-storage google-cloud-bigquery google-genai

In [ ]:
from pathlib import Path
from urllib.request import urlopen

# Nos notebooks do BigQuery Studio só o .ipynb é enviado, então trazemos o módulo do crawler
# do repositório da aula. É o mesmo recurso que a aula 1 usa para baixar os dados.
modulo = Path("simple_crawler.py")

if modulo.exists():
    print(f"{modulo} já está aqui.")
else:
    url = "https://raw.githubusercontent.com/robertogyn19/aula-pdm-pubsub/main/simple_crawler.py"
    modulo.write_bytes(urlopen(url).read())
    print(f"{modulo} baixado de {url}")

In [ ]:
import google.auth

# O projeto vem da credencial do ambiente, como na aula 1.
_, project_id = google.auth.default()
print(project_id)

## 1. De onde vem o anúncio

Os anúncios da aula 1 chegaram prontos, num `.zip`. Eles saíram da API pública de listagem do
[Chaves na Mão](https://www.chavesnamao.com.br), e é essa coleta que a gente refaz agora — o pipeline
começa aqui, não no arquivo.

O código vive em `simple_crawler.py`, e é ele que este notebook importa. Assim existe uma única versão
da lógica, e o que você executa aqui é exatamente o que gerou os arquivos de `dados/anuncios`.

In [ ]:
from simple_crawler import ChavesNaMaoCrawler

crawler = ChavesNaMaoCrawler()

### 1.1. Uma página da API

O site expõe a listagem em JSON. A URL combina dois níveis de navegação — o tipo de negócio (`level1`)
e a localidade (`level2`) — mais o número da página.

In [ ]:
level1 = "imoveis-a-venda"
level2 = "go-goiania"

url = f"{crawler.base_url}{crawler.base_path}?level1={level1}&level2={level2}&pg=1"
url

In [ ]:
payload = crawler.coletar_pagina(url)
list(payload.keys())

### 1.2. A estrutura da resposta

A resposta tem duas partes: `items`, com os anúncios da página, e `metadata`, com as informações de
paginação. É o `metadata` que diz quantas páginas existem e qual é a próxima URL.

In [ ]:
print("anúncios nesta página:", len(payload["items"]))
print("total de páginas:", payload["metadata"]["totalPages"])
print("próxima página:", payload["metadata"]["links"]["nextApiParams"])

### 1.3. Do JSON para o modelo

O JSON da API é aninhado e irregular: preço, área e contagem de quartos aparecem em formatos diferentes
conforme o anúncio. O `extrair_dados` achata isso num modelo `Anuncio` do Pydantic, com tipos fixos — e
é esse formato achatado que a tabela `aula_pdm.anuncios` espera.

Nem todo item vira anúncio: os que não têm `id` são descartados.

In [ ]:
anuncios = crawler.extrair_dados(payload)
print(f"{len(anuncios)} anúncios extraídos de {len(payload['items'])} itens")

print(anuncios[0].model_dump_json(indent=2))

In [ ]:
# O modelo também tem comportamento, não só dados
anuncios[0].endereco()

### 1.4. Paginação

O `coletar_paginas` repete o processo acima seguindo a próxima URL de cada resposta, com uma pausa entre
as requisições. A API não deixa passar da página 100, e o método já trata esse limite.

O `ultima` abaixo está fixo em 2 de propósito: sem ele, a coleta iria até o fim das mais de mil páginas.

In [ ]:
anuncios = crawler.coletar_paginas(level1, level2, primeira=1, ultima=2)
len(anuncios)

### 1.5. Como os arquivos da aula 1 foram gerados

O arquivo `dados/chavesnamao_level2.txt` tem a lista de estados e cidades usada na coleta. A função
`realizar_coleta_varias_paginas()`, no fim do `simple_crawler.py`, percorre essa lista e grava um
`.jsonl` por localidade em `dados/anuncios`, pulando os que já existem.

Foi assim que os 162 arquivos do `dados/anuncios.zip` foram produzidos, em setembro de 2025. Para
refazer a coleta, rode o script direto pelo terminal:

```bash
python simple_crawler.py
```

### 1.6. O anúncio que vamos seguir

Daqui em diante o notebook acompanha **um** anúncio. Escolhemos um que tenha algumas fotos, porque é
delas que a seção 4 vai precisar.

In [ ]:
anuncio = next(a for a in anuncios if len(a.imagens) >= 3)

print(anuncio.id, "|", anuncio.titulo)
print(anuncio.endereco())
print(anuncio.preco_fmt, "|", anuncio.quartos, "quartos |", len(anuncio.imagens), "imagens")

## 2. O caminho do texto: Pub/Sub → BigQuery

Este é o caminho que a aula 1 já deixou pronto. O tópico existe, a assinatura do BigQuery existe, e a
tabela existe. Não há nada para criar: basta publicar.

Repare no que **não** aparece aqui — nenhuma linha de `INSERT`, nenhuma conexão com o BigQuery. Quem
escreve na tabela é a assinatura, e ela já estava lá esperando.

In [ ]:
from google.cloud import pubsub_v1

topico_anuncios = f"projects/{project_id}/topics/aula-pdm-anuncios"

publisher = pubsub_v1.PublisherClient()
topico_anuncios

In [ ]:
# O model_dump_json() do Pydantic produz exatamente o formato que o esquema da tabela espera
futures = [
    publisher.publish(topico_anuncios, data=a.model_dump_json().encode("utf-8"))
    for a in anuncios
]

for fut in futures:
    fut.result(timeout=60)

print(f"{len(futures)} anúncios publicados em {topico_anuncios}")

### 2.1. Conferindo no BigQuery

A assinatura leva alguns instantes para escrever. Se o resultado vier vazio, espere um pouco e rode de
novo — e, se continuar vazio, o caminho de investigação é a DLQ da seção 5 da aula 1.

In [ ]:
from google.cloud import bigquery

cliente_bq = bigquery.Client(project=project_id)

sql = f"""
SELECT cidade, COUNT(*) AS anuncios, ROUND(AVG(preco), 2) AS preco_medio
FROM `{project_id}.aula_pdm.anuncios`
GROUP BY cidade
ORDER BY anuncios DESC
LIMIT 5
"""

for linha in cliente_bq.query(sql).result():
    print(f"{linha.cidade:25s} {linha.anuncios:6d}  {linha.preco_medio}")

A partir daqui o texto já é território conhecido: essa tabela é a camada Bronze, e as
camadas Silver e Gold saem dela com o SQL que vocês já viram com o professor Sávio.

O que a tabela **não** tem é o que só a foto mostra. É o outro caminho.

## 3. O caminho da imagem: download → Cloud Storage

A API não devolve as imagens, devolve o caminho de cada arquivo. A URL completa se monta juntando esse
caminho a uma base do site, e é isso que o `urls_imagens()` faz.

In [ ]:
for url_imagem in anuncio.urls_imagens():
    print(url_imagem)

### 3.1. Baixando as imagens

O `baixar_imagem` cuida de um detalhe chato: mesmo pedindo JPEG, o site às vezes responde WebP. Ele
detecta o formato pelo conteúdo e converte quando precisa, para todo arquivo daqui em diante ser JPEG.

In [ ]:
from simple_crawler import baixar_imagem

diretorio = Path(f"imagens/{anuncio.id}")

arquivos = [
    baixar_imagem(url_imagem, diretorio / f"{idx}.jpg")
    for idx, url_imagem in enumerate(anuncio.urls_imagens(), start=1)
]

print(f"{len(arquivos)} imagens em {diretorio}")

### 3.2. Enviando para o Cloud Storage

O bucket é o mesmo da seção 7 da aula 1. A célula abaixo o cria caso você não tenha feito aquela seção;
se ele já existir, não faz nada.

In [ ]:
bucket_name = f"{project_id}-aula-pdm"

!gcloud storage buckets describe gs://{bucket_name} --format="value(name)" 2>/dev/null || gcloud storage buckets create gs://{bucket_name} --location us-central1

In [ ]:
from google.cloud import storage

bucket = storage.Client(project=project_id).bucket(bucket_name)

for arquivo in arquivos:
    blob = bucket.blob(f"imagens/{anuncio.id}/{arquivo.name}")
    blob.upload_from_filename(arquivo)
    print(f"gs://{bucket_name}/{blob.name}")

As imagens agora estão no mesmo lugar em que a assinatura do GCS da aula 1 grava os
arquivos Avro, num prefixo diferente. Dois formatos, dois caminhos, um bucket.